# 01 合成価格を取得し、時刻と品質を確認する

## Goal

共通サービスの合成デモを明示的に生成し、価格の足終端、利用可能時刻、観測時刻、品質を確かめます。外部通信と口座ファイルの読取はありません。実プロバイダの取得手順は market-research/docs/DATA.md を参照してください。

## Setup

- 入力: market_research.services.build_demo_run() が作る固定の架空日足
- 通貨と調整方式: USD / raw。時刻: UTC
- 判断時刻: 足終端の1時間後。未来の足を判断に利用しません
- 出典: synthetic-demo。実市場のデータ品質や過去のPIT保存を証明するものではありません
- このノートは単独で再実行できます。ファイル出力はありません

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from market_research.services import build_demo_run

run = build_demo_run()
as_of = run.prices.index[-1].to_pydatetime()
print(f"mode={run.mode}, run_id={run.run_id}, input_sha256={run.input_hash}")
print(f"as_of={as_of.isoformat()}, observations={len(run.prices)}, assets={len(run.prices.columns)}")

## Steps

### 1. 足の来歴と利用可能時刻を読む

表は冒頭6本だけ表示します。bar_end は足の終端、available_at はその値を使用できる時刻、observed_at は取得した時刻です。

In [ ]:
bars = pd.DataFrame(
    {
        "instrument_id": bar.instrument.instrument_id,
        "bar_end_utc": bar.bar_end,
        "available_at_utc": bar.available_at,
        "observed_at_utc": bar.observed_at,
        "close_usd": bar.close,
        "final": bar.is_final,
        "quality": bar.quality,
        "provider": bar.provider,
        "adjustment": bar.adjustment,
    }
    for bar in run.bars
)
bars.head(6)

### 2. 同じ判断時刻に利用可能な2銘柄を比べる

各系列を初日の値で100に基準化します。縦軸はUSDの価格ではなく指数です。架空の60営業日だけを表示します。

In [ ]:
normalized = run.prices.div(run.prices.iloc[0]).mul(100)
figure = go.Figure()
for asset, dash in zip(normalized.columns, ("solid", "dash"), strict=True):
    figure.add_scatter(
        x=normalized.index,
        y=normalized[asset],
        mode="lines",
        name=asset,
        line={"dash": dash},
    )
figure.update_layout(
    title="合成デモ価格: 初日の判断時刻を100に基準化",
    xaxis_title="判断時刻 (UTC)",
    yaxis_title="基準化指数 (初日=100)",
    template="plotly_white",
)
figure

## Checks

足はすべて確定値で、利用可能時刻と観測時刻は足終端より後です。下の検査は合成入力の整合性だけを確認します。

In [ ]:
assert len(run.bars) == len(run.prices) * len(run.prices.columns)
assert bars["final"].all()
assert bars["quality"].eq("ok").all()
assert (bars["bar_end_utc"] <= bars["available_at_utc"]).all()
assert (bars["available_at_utc"] <= bars["observed_at_utc"]).all()
assert (bars["observed_at_utc"] <= as_of).all()
assert bars["provider"].eq("synthetic-demo").all()
assert bars["adjustment"].eq("raw").all()
print("足120本、60判断時点、2銘柄: 時刻・品質の検査を通過")

## Next Steps

02ではこの合成足を一時的な保存snapshotにして、信号を時点別に作ります。現在から過去を一括取得した系列を、過去に利用できたPIT系列と呼ばないでください。